In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print("\nDate columns available:")
print([col for col in df.columns if 'date' in col.lower()])

Rows: 180,519, Columns: 53

Date columns available:
['order date (DateOrders)', 'shipping date (DateOrders)']


In [15]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"After removing duplicates: {df_clean.shape[0]:,} rows")

After removing duplicates: 180,519 rows


In [17]:
df_clean['Shipping date (DateOrders)'] = pd.to_datetime(
    df_clean['shipping date (DateOrders)'], 
    errors='coerce',
    format='mixed'
)

df_clean['Days for shipping (real)'] = pd.to_numeric(
    df_clean['Days for shipping (real)'], 
    errors='coerce'
)
df_clean['Days for shipment (scheduled)'] = pd.to_numeric(
    df_clean['Days for shipment (scheduled)'], 
    errors='coerce'
)

print("✓ Data types converted successfully")

✓ Data types converted successfully


In [18]:
df_clean['Delay_Days'] = (
    df_clean['Days for shipping (real)'] - 
    df_clean['Days for shipment (scheduled)']
)

df_clean['Is_Late'] = (df_clean['Delay_Days'] > 0).astype(int)

df_clean['Order_YearMonth'] = df_clean['Shipping date (DateOrders)'].dt.to_period('M')

print("✓ New columns created:")
print(f"  - Delay_Days")
print(f"  - Is_Late")
print(f"  - Order_YearMonth")

✓ New columns created:
  - Delay_Days
  - Is_Late
  - Order_YearMonth


In [20]:
print("DATA QUALITY CHECK:")
print("="*60)
print(f"\nMissing values:")
key_cols = ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Delay_Days', 'Is_Late']
print(df_clean[key_cols].isnull().sum())

print(f"\n\nDelay_Days Statistics:")
print(df_clean['Delay_Days'].describe())

print(f"\n\nIs_Late Distribution:")
print(df_clean['Is_Late'].value_counts())

print(f"\n\nSummary:")
print(f"Late deliveries: {df_clean['Is_Late'].sum():,} ({df_clean['Is_Late'].mean()*100:.1f}%)")
print(f"On-time deliveries: {(1-df_clean['Is_Late']).sum():,} ({(1-df_clean['Is_Late']).mean()*100:.1f}%)")

DATA QUALITY CHECK:

Missing values:
Days for shipping (real)         0
Days for shipment (scheduled)    0
Delay_Days                       0
Is_Late                          0
dtype: int64


Delay_Days Statistics:
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Delay_Days, dtype: float64


Is_Late Distribution:
Is_Late
1    103400
0     77119
Name: count, dtype: int64


Summary:
Late deliveries: 103,400 (57.3%)
On-time deliveries: 77,119 (42.7%)


In [23]:
import os

# Create processed data folder
os.makedirs('../data/processed', exist_ok=True)
print("✓ /data/processed/ folder created")

# Now save the cleaned data
df_clean.to_csv('../data/processed/cleaned_supply_chain_data.csv', index=False)
print("✓ Cleaned data saved successfully!")
print(f"✓ Final shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

✓ /data/processed/ folder created
✓ Cleaned data saved successfully!
✓ Final shape: 180,519 rows × 57 columns


In [24]:
# Verify the file was saved
import os
file_path = '../data/processed/cleaned_supply_chain_data.csv'
if os.path.exists(file_path):
    print("✓ File exists!")
    print(f"✓ File size: {os.path.getsize(file_path) / (1024*1024):.2f} MB")
else:
    print("✗ File not found")

✓ File exists!
✓ File size: 97.93 MB
